In [ ]:
import math, torch, numpy as np, trimesh
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
class InsidePanoramaCamera:
    def __init__(self, resolution, radius=1.0, device="cpu", dtype=torch.float32):
        self.H, self.W = resolution, 2 * resolution
        self.R = float(radius)
        self.device, self.dtype = torch.device(device), dtype
        self.scene_center = torch.zeros(3, device=self.device, dtype=self.dtype)

    @torch.no_grad()
    def get_rays(self):
        H, W, R, d = self.H, self.W, self.R, self.device
        phi   = (torch.arange(H, device=d) + 0.5) / H * math.pi
        theta = (torch.arange(W, device=d) + 0.5) / W * 2 * math.pi
        phi, theta = torch.meshgrid(phi, theta, indexing="ij")

        outward = torch.stack(
            (torch.sin(phi) * torch.cos(theta),
             torch.cos(phi),
             torch.sin(phi) * torch.sin(theta)), -1)          # (H,W,3)

        origins    = self.scene_center + R * outward           # (H,W,3)
        directions = (self.scene_center - origins) / R         # (H,W,3)
        return origins, directions


In [ ]:
mesh_path = "demo_data/cow_mesh/cow.obj"
# mesh_path = "demo_data/pipe/fb42332b3f5e491cb0c4b5ba7ed6f374.obj"
mesh = trimesh.load(mesh_path, process=False)      # keeps UV + texture
# if isinstance(mesh.visual, trimesh.visual.texture.TextureVisuals):
#     mesh.visual = mesh.visual.to_color()           # bake texture → per-vertex RGB

In [ ]:
R = 5
camera = InsidePanoramaCamera(resolution=256, radius=R, device="cpu")
orig_t, dir_t = camera.get_rays()                  # torch (H,W,3)
H, W = orig_t.shape[:2]

# Trimesh wants NumPy (N,3)
orig_np = orig_t.reshape(-1, 3).cpu().numpy()
dir_np  = dir_t.reshape(-1, 3).cpu().numpy()


In [ ]:
import math, numpy as np
from tqdm import tqdm

# ─────────────────── parameters ──────────────────────────────────────
process_batch_size = 200
max_hits           = 4                   # keep at most K hits per ray
num_rays           = orig_np.shape[0]
num_batches        = math.ceil(num_rays / process_batch_size)

# ─────────────────── allocate K slots with sentinel values ───────────
locs_dict      = {k: np.full((num_rays, 3),  np.nan, dtype=np.float32) for k in range(max_hits)}
dists_dict     = {k: np.full(num_rays,       np.inf, dtype=np.float32) for k in range(max_hits)}
idx_tris_dict  = {k: np.full(num_rays,       -1,     dtype=np.int32)   for k in range(max_hits)}
idx_rays_dict  = {k: np.full(num_rays,       -1,     dtype=np.int32)   for k in range(max_hits)}
hits_seen = np.zeros(num_rays, dtype=np.int32)   # current slot per ray

# ─────────────────── batch-wise intersection ─────────────────────────
for i in tqdm(range(num_batches)):
    start, end = i * process_batch_size, min((i + 1)*process_batch_size, num_rays)

    # all intersections for this mini-batch
    b_locs, b_idx_rays, b_idx_tris = mesh.ray.intersects_location(
        ray_origins    = orig_np[start:end],
        ray_directions = dir_np[start:end],
        multiple_hits  = True,
    )

    # convert local → global ray IDs
    b_idx_rays += start

    # depth along each ray
    b_dists = np.linalg.norm(b_locs - orig_np[b_idx_rays], axis=1)

    # stable sort by (rayID, depth)
    order = np.lexsort((b_dists, b_idx_rays))
    b_locs, b_dists = b_locs[order], b_dists[order]
    b_idx_rays, b_idx_tris = b_idx_rays[order], b_idx_tris[order]

    # scatter hits into the first K free slots
    for loc, dist, r, tri in zip(b_locs, b_dists, b_idx_rays, b_idx_tris):
        k = hits_seen[r]
        if k >= max_hits:
            continue                       # already stored K hits for this ray
        locs_dict[k][r]     = loc
        dists_dict[k][r]    = dist
        idx_tris_dict[k][r] = tri
        idx_rays_dict[k][r] = r                 # store the ray-ID explicitly
        hits_seen[r]  += 1


In [ ]:
dists_dict[1].min()

In [ ]:
def get_shaded_image(mesh, locs, idx_rays, idx_tris, device='cpu'):
    # convert back to Torch for shading
    locs_t     = torch.as_tensor(locs,     dtype=torch.float32)
    idx_rays_t = torch.as_tensor(idx_rays, dtype=torch.long)
    idx_tris_t = torch.as_tensor(idx_tris, dtype=torch.long)

    # ---- gather triangle data ------------------------------------------
    verts   = torch.as_tensor(mesh.vertices, dtype=torch.float32, device=device)
    faces_i = torch.as_tensor(mesh.faces,    dtype=torch.long,    device=device)

    # either per-vertex colours (ColorVisuals) or None (TextureVisuals)
    vcol = (torch.as_tensor(mesh.visual.vertex_colors[:, :3],
                            dtype=torch.float32, device=device)/255.0
            if hasattr(mesh.visual, "vertex_colors") and
            mesh.visual.vertex_colors is not None else None)

    # vertex normals (compute if missing)
    if mesh.vertex_normals is None or len(mesh.vertex_normals) == 0:
        mesh.vertex_normals = mesh.vertex_normals
    vnorm = torch.as_tensor(mesh.vertex_normals,
                            dtype=torch.float32, device=device)

    tris_verts = verts[faces_i[idx_tris_t]]               # (M,3,3)
    tris_norm  = vnorm[faces_i[idx_tris_t]]               # (M,3,3)

    # ---- barycentric weights -------------------------------------------
    v0, v1, v2 = tris_verts.unbind(1)
    v0v1, v0v2 = v1 - v0, v2 - v0
    d00 = (v0v1 * v0v1).sum(1); d01 = (v0v1 * v0v2).sum(1); d11 = (v0v2 * v0v2).sum(1)
    d20 = ((locs_t - v0) * v0v1).sum(1); d21 = ((locs_t - v0) * v0v2).sum(1)
    denom = d00 * d11 - d01 * d01
    v = (d11 * d20 - d01 * d21) / denom
    w = (d00 * d21 - d01 * d20) / denom
    u = 1 - v - w
    bary = torch.stack((u, v, w), 1).clamp_(0, 1)          # (M,3)

    # -------------------------------------------------------------------- #
    # (A)  TextureVisuals  → bilinear UV sampling                          #
    # -------------------------------------------------------------------- #
    if isinstance(mesh.visual, trimesh.visual.texture.TextureVisuals):
        tex_img = mesh.visual.material.image.convert("RGB")
        tex_np  = np.asarray(tex_img)[..., :3]
        tex_map = (torch.from_numpy(tex_np)
                        .float()
                        .permute(2, 0, 1)            # (3,Ht,Wt)
                        .unsqueeze(0)                # (1,3,Ht,Wt)
                        .to(device)) / 255.0

        # mesh.visual.uv : (F, 3, 2)  — one UV per vertex per face
        verts_uv = torch.as_tensor(mesh.visual.uv,
                                dtype=torch.float32, device=device)  # (V,2)
        tris_uv  = verts_uv[faces_i[idx_tris_t]] 
        uvs      = (tris_uv * bary[..., None]).sum(1)  # (M,2)

        # grid-sample (bilinear)
        grid = torch.stack((uvs[:, 0] * 2 - 1,
                            (1 - uvs[:, 1]) * 2 - 1), 1)   # (M,2) in [-1,1]
        rgb_tex = torch.nn.functional.grid_sample(
            tex_map, grid.view(1, -1, 1, 2),
            mode="bilinear", align_corners=True
        ).view(3, -1).t()                                # (M,3)
    else:
        # ---------------------------------------------------------------- #
        # (B)  ColorVisuals  → barycentric per-vertex colour               #
        # ---------------------------------------------------------------- #
        tris_col = vcol[faces_i[idx_tris_t]]                # (M,3,3)
        rgb_tex  = (tris_col * bary[..., None]).sum(1)      # (M,3)

    # ---- interpolate vertex normals (Phong) -----------------------------
    normals = (tris_norm * bary[..., None]).sum(1)
    normals = torch.nn.functional.normalize(normals, dim=1)   # (M,3)

    # ---- Phong lighting -------------------------------------------------
    view_dir  = torch.nn.functional.normalize(
        -dir_t.reshape(-1, 3)[idx_rays_t], dim=1)
    light_dir = view_dir                                       # head-on light

    cos_theta = torch.clamp((normals * light_dir).sum(1, keepdim=True), 0, 1)
    kd, ks, shininess = 1.0, 0.25, 64

    reflect_dir = 2 * cos_theta * normals - light_dir
    cos_alpha   = torch.clamp((reflect_dir * view_dir).sum(1, keepdim=True), 0, 1)
    spec        = ks * (cos_alpha ** shininess)

    # shaded = 0.1 * rgb_tex + kd * cos_theta * rgb_tex + spec   # ambient + diff + spec
    shaded = 0.1 * rgb_tex + kd * cos_theta.abs() * rgb_tex

    return shaded


In [ ]:
hit_idx = 0
shaded = get_shaded_image(mesh, locs_dict[hit_idx], idx_rays_dict[hit_idx], idx_tris_dict[hit_idx], device='cpu')
idx_rays_t = torch.as_tensor(idx_rays_dict[hit_idx], dtype=torch.long)

# start with black background
image = torch.zeros(H * W, 3)
image[idx_rays_t] = shaded.cpu()        # fill only the rays that hit
image = image.view(H, W, 3).clamp(0, 1).numpy()

plt.figure(figsize=(10,5))
plt.imshow(image)
plt.show()


In [ ]:
hit_idx = 1
shaded = get_shaded_image(mesh, locs_dict[hit_idx], idx_rays_dict[hit_idx], idx_tris_dict[hit_idx], device='cpu')
idx_rays_t = torch.as_tensor(idx_rays_dict[hit_idx], dtype=torch.long)

# start with black background
image = torch.zeros(H * W, 3)
image[idx_rays_t] = shaded.cpu()        # fill only the rays that hit
image = image.view(H, W, 3).clamp(0, 1).numpy()

plt.figure(figsize=(10,5))
plt.imshow(image)
plt.show()


### ================